In [ ]:
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")   # IMPORTANT for PYNQ (no display)
import matplotlib.pyplot as plt
import os
import datetime
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import ImageReader

# ─────────────────────────────────────────────
# SETTINGS
# ─────────────────────────────────────────────
OUTDIR = "/home/xilinx/report"
os.makedirs(OUTDIR, exist_ok=True)
epsilon = 1e-6

# ─────────────────────────────────────────────
# NORMALIZATION
# ─────────────────────────────────────────────
def norm(x):
    return (x - x.min()) / (x.max() - x.min() + epsilon)

# ─────────────────────────────────────────────
# MAIN REPORT FUNCTION
# ─────────────────────────────────────────────
def generate_report(bands):

    # Extract bands (31-band input)
    blue    = bands[:, :, 7]
    green   = bands[:, :, 16]
    red     = bands[:, :, 26]
    rededge = bands[:, :, 29]
    nir     = bands[:, :, 30]

    # Normalize
    blue, green, red, rededge, nir = map(norm, [blue, green, red, rededge, nir])

    # RGB image
    rgb = np.stack([red, green, blue], axis=-1)

    # ─────────────────────────────
    # Biomarkers
    # ─────────────────────────────
    abs_red     = np.log(1 / (red + epsilon))
    abs_green   = np.log(1 / (green + epsilon))
    abs_nir     = np.log(1 / (nir + epsilon))
    abs_rededge = np.log(1 / (rededge + epsilon))

    HbO2 = 0.65 * abs_nir + 0.25 * abs_rededge + 0.10 * abs_green
    Hb   = 0.70 * abs_red + 0.30 * abs_green
    StO2 = HbO2 / (HbO2 + Hb + epsilon)
    Perf = abs_green

    # Risk model
    Risk = 0.5*(1 - norm(StO2)) + 0.3*(1 - norm(Perf)) + 0.2*norm(abs_rededge)
    Risk_n = norm(Risk)

    # ─────────────────────────────
    # Zone Analysis
    # ─────────────────────────────
    h = Risk.shape[0]

    toe  = Risk_n[:h//3, :]
    mid  = Risk_n[h//3:2*h//3, :]
    heel = Risk_n[2*h//3:, :]

    toe_r  = float(np.mean(toe))
    mid_r  = float(np.mean(mid))
    heel_r = float(np.mean(heel))
    mean_r = float(np.mean(Risk_n))

    # Assessment
    if mean_r < 0.5:
        status = "NORMAL"
        color = "green"
    else:
        status = "HIGH RISK"
        color = "red"

    # ─────────────────────────────
    # FIGURE 1: Risk Map
    # ─────────────────────────────
    plt.figure(figsize=(6,4))
    plt.imshow(Risk_n, cmap="jet")
    plt.title("Ulcer Risk Map")
    plt.colorbar()
    plt.axis("off")
    plt.savefig(f"{OUTDIR}/risk.png", dpi=120)
    plt.close()

    # ─────────────────────────────
    # FIGURE 2: RGB
    # ─────────────────────────────
    plt.figure(figsize=(6,4))
    plt.imshow(rgb)
    plt.title("RGB Image")
    plt.axis("off")
    plt.savefig(f"{OUTDIR}/rgb.png", dpi=120)
    plt.close()

    # ─────────────────────────────
    # FIGURE 3: Zones
    # ─────────────────────────────
    plt.figure(figsize=(6,4))
    plt.imshow(Risk_n, cmap="jet")
    plt.axhline(h//3, color='white')
    plt.axhline(2*h//3, color='white')
    plt.title("Zone-wise Risk")
    plt.savefig(f"{OUTDIR}/zones.png", dpi=120)
    plt.close()

    # ─────────────────────────────
    # PDF GENERATION
    # ─────────────────────────────
    pdf_path = f"{OUTDIR}/report.pdf"
    c = rl_canvas.Canvas(pdf_path, pagesize=A4)

    # Page 1
    c.setFont("Helvetica-Bold", 16)
    c.drawString(150, 800, "Diabetic Foot Report")

    c.setFont("Helvetica", 12)
    c.drawString(50, 750, f"Assessment: {status}")
    c.drawString(50, 720, f"Mean Risk: {mean_r:.3f}")
    c.drawString(50, 690, f"Toe Risk: {toe_r:.3f}")
    c.drawString(50, 660, f"Mid Risk: {mid_r:.3f}")
    c.drawString(50, 630, f"Heel Risk: {heel_r:.3f}")

    date = datetime.datetime.now().strftime("%d-%m-%Y %H:%M")
    c.drawString(50, 600, f"Generated: {date}")

    c.drawImage(ImageReader(f"{OUTDIR}/rgb.png"), 50, 400, width=200, height=150)
    c.drawImage(ImageReader(f"{OUTDIR}/risk.png"), 300, 400, width=200, height=150)

    c.showPage()

    # Page 2
    c.drawImage(ImageReader(f"{OUTDIR}/zones.png"), 50, 400, width=400, height=250)

    c.save()

    print("✅ Report Generated:", pdf_path)